# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Clustering.

My question is "what kinds of pages exist across the content inventory?" — per the framing skill's mapping table, that phrasing ("what kinds of items exist?") maps directly to clustering, not classification (no predefined label to sort into), not ranking/scoring (I'm not producing one ordered priority list), and not signal analysis (I'm not testing which individual signals correlate with an outcome). I'm grouping pages by similarity across several metrics at once  position, engagement, freshness, word count, volume  to find structure, not to predict a known answer.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is no target label in the supervised sense  clustering is unsupervised, so nothing is being predicted from a known answer. What I define instead is the feature set the clusters get built from (avg_position, engagement_rate, word_count, impressions_90d, content_age_days, etc.). The cluster assignment that comes out afterward is a derived grouping, not an observed outcome and not a hand-written rule  it's an output of the algorithm's own similarity logic, which I then interpret and name.

One explicit exclusion, per the data skill's label-trap warning: trend_direction and trend_pct are never used as clustering features, since trend_direction is itself computed from trend_pct  including either would let a pre-existing rule quietly shape the grouping instead of letting the raw signals speak for themselves.

In [19]:
print("Features used for clustering:")
print(feature_cols)
# This confirms the features selected based on your explanation, excluding 'trend_direction' and 'trend_pct'.

Features used for clustering:
['avg_position', 'engagement_rate', 'word_count', 'impressions_90d', 'search_volume', 'competition', 'content_age_days']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Silhouette score, paired with a manual sanity check.

Silhouette score measures, for each page, whether it sits closer to its own cluster's center or to a neighboring cluster's center, averaged across all pages, on a scale from -1 to 1. A score meaningfully above 0 I'll use >0.25 as my working "good" threshold, to be calibrated once I see the real distribution  means the clusters are genuinely separated groups rather than an arbitrary slice through one continuous blob.

Silhouette score alone can be gamed by a degenerate clustering (e.g. one giant cluster plus a few tiny outlier clusters still scores deceptively well), so I'll pair it with pulling 5–10 real pages from each cluster and confirming by eye that they intuitively belong together before trusting the number

In [18]:
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd

# Assuming 'df' is your processed DataFrame from the previous steps
# Select numerical features relevant for clustering
# Example: feature_cols = ['avg_position', 'engagement_rate', 'word_count', 'impressions_90d']
# Make sure to handle any missing values in your selected features before scaling
# Corrected feature_cols by removing non-existent columns identified from df.columns
feature_cols = ['avg_position', 'engagement_rate', 'word_count', 'impressions_90d', 'search_volume', 'competition', 'content_age_days']

# Drop rows with NaN values in selected features for simplicity, or impute them
df_features = df[feature_cols].dropna()

# Scale the features (important for distance-based clustering algorithms)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)

# --- Placeholder for Clustering Algorithm ---
# You would apply your chosen clustering algorithm here (e.g., KMeans, DBSCAN, AgglomerativeClustering)
# For example:
n_clusters = 3 # You can change the number of clusters
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10) # n_init is set to suppress future warnings
cluster_labels = kmeans.fit_predict(X_scaled)

# Add cluster labels back to the original (filtered and feature-selected) DataFrame
df_features_clustered = df_features.copy()
df_features_clustered['cluster_label'] = cluster_labels

print(f"KMeans clustering performed with {n_clusters} clusters.")
print("DataFrame with cluster labels (first 5 rows):")
display(df_features_clustered.head())

# Once you have cluster_labels, calculate the Silhouette Score
print(f"Silhouette Score: {silhouette_score(X_scaled, cluster_labels):.3f}")

KMeans clustering performed with 3 clusters.
DataFrame with cluster labels (first 5 rows):


,avg_position,engagement_rate,word_count,impressions_90d,search_volume,competition,content_age_days,cluster_label
0,10.6,5.88,3221.0,3803,10.0,0.67,187,0
1,20.3,0.00,2481.0,15320,90.0,0.01,445,2
2,36.5,0.00,3515.0,12581,0.0,0.00,141,1
4,44.0,0.00,2803.0,19140,0.0,0.00,263,1
5,8.5,0.00,3080.0,3970,720.0,1.00,147,0


Silhouette Score: 0.289


In [15]:
print(df.columns)

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (content_id). Grain is verified with a probe below, not assumed — per the data skill's rule, a blind drop_duplicates can silently discard meaningful rows if content_id isn't actually unique.

In [13]:
import pandas as pd

# Utiliser le DataFrame déjà chargé depuis GitHub
raw = df_from_github.copy()
print("Raw rows:", len(raw))

# Grain probe — confirm one row per content_id, don't assume it
grain_check = raw.groupby("content_id").size()
violations = grain_check[grain_check > 1]
print("content_ids with more than 1 row:", len(violations))
if len(violations) > 0:
    print(violations.head())

# Apply the lane guide's filter rule
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
df_position = df[df["avg_position"] > 0]  # avg_position == 0 means "no data," not rank zero

print("\nRows after impressions/age filter:", len(df))

# Show the actual unit of analysis — one row = one page's metric profile
df[["content_id", "avg_position", "engagement_rate", "word_count", "impressions_90d"]].head()

Raw rows: 30000
content_ids with more than 1 row: 0

Rows after impressions/age filter: 30000


,content_id,avg_position,engagement_rate,word_count,impressions_90d
0,content_304f48230142,10.6,5.88,3221.0,3803
1,content_a1fb4e703a9e,20.3,0.00,2481.0,15320
2,content_9aa793d4d895,36.5,0.00,3515.0,12581
3,content_331d6c4de07b,6.2,1.28,NaN,11751
4,content_d99b7a2d90ca,44.0,0.00,2803.0,19140


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-statement rule requires deciding in advance which combinations of conditions define a group — e.g. "if position < 10 AND engagement < 0.3%, call it archetype X." That works when one or two dimensions matter, but page archetypes here likely depend on interactions across many metrics simultaneously: position, engagement, freshness, word count, and impression volume. A page can be low-position but high-engagement, or high-impressions but stale — the useful groupings aren't obvious ahead of time, and an if-statement chain would need dozens of hand-guessed nested branches to capture them, with someone guessing the right thresholds for every combination.

Clustering instead lets the data find which pages actually sit close together across all dimensions at once, surfacing groupings nobody would have thought to hard-code in advance — that's the concrete justification for ML over a rulebook here, not just "ML sounds more advanced."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.